In [9]:
!pip install grep

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for grep: filename=grep-0.3.2-py3-none-any.whl size=2957 sha256=5dca587893f92fe6b1785e250717ad6f8febc97a4c6f56ba69c97a391ffb8ff9
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\86\8a\c7\dc9aabd700c3cf8a033feda1b7fe5a39250f254de8e8636996
Successfully built grep


In [13]:
pip freeze | findstr scikit-learn


scikit-learn==1.7.0
Note: you may need to restart the kernel to use updated packages.


In [14]:
!python -V

Python 3.13.5


In [15]:
import pickle
import pandas as pd

In [16]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [42]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [51]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-05.parquet')

In [52]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [53]:
import numpy as np

In [54]:
y_pred.mean()

np.float64(14.242595513316312)

In [46]:
np.std(y_pred)

np.float64(6.353996941249665)

In [34]:
year = 2023
month = 4

input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet'
output_file = f'output/yellow_tripdata_{year:04d}-{month:02d}.parquet'

In [35]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [36]:
!mkdir output

A subdirectory or file output already exists.


In [27]:
df = read_data(input_file)
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [28]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration,ride_id
0,2,2023-03-01 00:06:43,2023-03-01 00:16:43,1.0,0.00,1.0,N,238,42,2,...,1.0,0.5,0.00,0.0,1.0,11.10,0.0,0.00,10.000000,2023/03_0
1,2,2023-03-01 00:08:25,2023-03-01 00:39:30,2.0,12.40,1.0,N,138,231,1,...,6.0,0.5,12.54,0.0,1.0,76.49,2.5,1.25,31.083333,2023/03_1
2,1,2023-03-01 00:15:04,2023-03-01 00:29:26,0.0,3.30,1.0,N,140,186,1,...,3.5,0.5,4.65,0.0,1.0,28.05,2.5,0.00,14.366667,2023/03_2
3,1,2023-03-01 00:49:37,2023-03-01 01:01:05,1.0,2.90,1.0,N,140,43,1,...,3.5,0.5,4.10,0.0,1.0,24.70,2.5,0.00,11.466667,2023/03_3
4,2,2023-03-01 00:08:04,2023-03-01 00:11:06,1.0,1.23,1.0,N,79,137,1,...,1.0,0.5,2.44,0.0,1.0,14.64,2.5,0.00,3.033333,2023/03_4


In [29]:
df_result = pd.DataFrame()
df_result['ride_id'] = df['ride_id']
df_result['predicted_duration'] = y_pred

In [30]:
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [33]:
!dir output

 Volume in drive C is OS-10
 Volume Serial Number is 84DC-5770

 Directory of C:\Users\HP\mlopszoomcamp\mlops-zoomcamp\cohorts\2024\04-deployment\homework\output

06/17/2025  09:50 PM    <DIR>          .
06/17/2025  09:50 PM    <DIR>          ..
06/17/2025  09:50 PM        68,641,024 yellow_tripdata_2023-03.parquet
               1 File(s)     68,641,024 bytes
               2 Dir(s)  100,266,713,088 bytes free
